# Assessment: PM2.5-Weighted vs Unweighted VAE (latent_dim=6)

Compare `latent6_pm25wt` (recon loss weighted by |corr(feature, log_pm25)|) vs `latent6_nodoy` (uniform weights).

Questions:
1. Did the weighting shift reconstruction quality toward PM2.5-relevant features?
2. Does the weighted latent space predict PM2.5 better?
3. What did the latent space give up in exchange?

In [6]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import json
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Use absolute paths so notebook runs from any cwd
base_dir = os.path.dirname(os.path.abspath('__file__'))
if not os.path.exists('embeddings.npz'):
    os.chdir('/home/vcaristo/pm_data/vae/runs/latent6_pm25wt')

# Load data
df = pd.read_parquet('/home/vcaristo/pm_data/svgp/full_conus/loso_temp_data.parquet')
y = np.log(df['pm25'].values + 1)

time_varying = ['aot', 'wind', 'hgt', 'cld', 'longwave', 'rh', 'tmax', 'smogI', 'smogP']
static = ['lat', 'lon', 'logpd2500g', 'minf_5000', 'sd50k', 'heavy_industrial_ind1', 'housing']
raw_cols = [f for f in time_varying if f in df.columns] + [f for f in static if f in df.columns]
X_raw = df[raw_cols].values

# Load both runs
runs_base = '/home/vcaristo/pm_data/vae/runs'
runs = {}
for name in ['latent6_nodoy', 'latent6_pm25wt']:
    emb = np.load(f'{runs_base}/{name}/embeddings.npz')
    with open(f'{runs_base}/{name}/config.json') as f:
        cfg = json.load(f)
    model_data = torch.load(f'{runs_base}/{name}/model.pt', map_location='cpu', weights_only=False)
    runs[name] = {
        'z_mu': emb['z_mu'],
        'assignments': emb['assignments'],
        'mix_weights': emb['mix_weights'],
        'cfg': cfg,
        'history': model_data['history'],
        'feature_cols': list(emb['feature_cols']),
    }

print(f"Loaded both runs. Shape: {runs['latent6_nodoy']['z_mu'].shape}")

Loaded both runs. Shape: (1883144, 6)


## Per-feature reconstruction comparison

Did weighting shift quality toward PM2.5-relevant features?

In [7]:
# Reconstruct from both models to get per-feature R²
import sys
sys.path.insert(0, '/home/vcaristo/pm_data/vae')
from train import GMMVAE

feature_cols = runs['latent6_nodoy']['feature_cols']
INPUT_DIM = len(feature_cols)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols].values).astype(np.float32)
X_tensor = torch.tensor(X_scaled)

# Use a subsample for reconstruction
np.random.seed(42)
sub_idx = np.random.choice(len(X_tensor), 200_000, replace=False)
X_sub = X_tensor[sub_idx]

r2_per_feat = {}
for name in ['latent6_nodoy', 'latent6_pm25wt']:
    cfg = runs[name]['cfg']
    model = GMMVAE(INPUT_DIM, cfg['latent_dim'], cfg['hidden_dims'], cfg['K'])
    state = torch.load(f'{runs_base}/{name}/model.pt', map_location='cpu', weights_only=False)['model_state_dict']
    model.load_state_dict(state)
    model.eval()
    
    with torch.no_grad():
        x_recon, _, _, _ = model(X_sub)
    x_recon = x_recon.numpy()
    x_orig = X_sub.numpy()
    
    mse = ((x_orig - x_recon) ** 2).mean(axis=0)
    r2 = 1 - mse / x_orig.var(axis=0)
    r2_per_feat[name] = r2

# Plot side by side
fig, ax = plt.subplots(figsize=(14, 6))
x_pos = np.arange(INPUT_DIM)
width = 0.35

bars1 = ax.bar(x_pos - width/2, r2_per_feat['latent6_nodoy'], width, label='Unweighted', alpha=0.8)
bars2 = ax.bar(x_pos + width/2, r2_per_feat['latent6_pm25wt'], width, label='PM2.5-weighted', alpha=0.8)

ax.set_xticks(x_pos)
ax.set_xticklabels(feature_cols, rotation=45, ha='right')
ax.set_ylabel('Reconstruction R²')
ax.set_title('Per-Feature Reconstruction: Unweighted vs PM2.5-Weighted')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('plots/recon_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print delta
print(f"{'Feature':<25} {'Unweighted':>10} {'Weighted':>10} {'Delta':>8} {'Weight':>8}")
print("-" * 65)
weights = runs['latent6_pm25wt']['cfg'].get('feature_weights', [1]*INPUT_DIM)
for i, f in enumerate(feature_cols):
    r2_uw = r2_per_feat['latent6_nodoy'][i]
    r2_wt = r2_per_feat['latent6_pm25wt'][i]
    print(f"{f:<25} {r2_uw:>10.4f} {r2_wt:>10.4f} {r2_wt - r2_uw:>+8.4f} {weights[i]:>8.2f}")

Feature                   Unweighted   Weighted    Delta   Weight
-----------------------------------------------------------------
aot                           0.5209     0.6133  +0.0925     1.63
wind                          0.5667     0.8156  +0.2489     2.93
hgt                           0.5508     0.8139  +0.2630     2.78
cld                           0.7757     0.8968  +0.1210     2.60
longwave                      0.8263     0.9082  +0.0819     2.75
rh                            0.7626     0.8601  +0.0975     2.72
tmax                          0.8303     0.9118  +0.0815     2.96
smogI                         0.7869     0.8773  +0.0905     1.35
smogP                         0.7777     0.9144  +0.1367     5.00
lat                           0.7372     0.9033  +0.1662     2.41
lon                           0.7763     0.8828  +0.1065     2.03
logpd2500g                    0.7764     0.9310  +0.1547     4.09
minf_5000                     0.6057     0.8661  +0.2604     3.33
sd50k     

## PM2.5 prediction from latent space

Does the weighted latent space predict PM2.5 better?

In [8]:
# Train/test split (same seed as predict_pm25.ipynb)
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)
y_train, y_test = y[idx_train], y[idx_test]

# Subsample for RF
np.random.seed(42)
n_sub = 200_000
sub_train = np.random.choice(idx_train, n_sub, replace=False)
sub_test = np.random.choice(idx_test, n_sub // 4, replace=False)

results = {}

# Raw features baseline
scaler_raw = StandardScaler()
X_train_s = scaler_raw.fit_transform(X_raw[idx_train])
X_test_s = scaler_raw.transform(X_raw[idx_test])

lr_raw = LinearRegression().fit(X_train_s, y_train)
results['LR raw (16 feat)'] = r2_score(y_test, lr_raw.predict(X_test_s))

# For each VAE run: LR and RF on latent space
for name, label in [('latent6_nodoy', 'unweighted'), ('latent6_pm25wt', 'pm25-weighted')]:
    Z = runs[name]['z_mu']
    
    # LR on full data
    lr = LinearRegression().fit(Z[idx_train], y_train)
    results[f'LR {label} (6D)'] = r2_score(y_test, lr.predict(Z[idx_test]))
    
    # RF on subsample
    rf = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
    rf.fit(Z[sub_train], y[sub_train])
    results[f'RF {label} (6D)'] = r2_score(y[sub_test], rf.predict(Z[sub_test]))

# RF on raw features (subsample)
X_raw_s = StandardScaler().fit_transform(X_raw)
rf_raw = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
rf_raw.fit(X_raw_s[sub_train], y[sub_train])
results['RF raw (16 feat)'] = r2_score(y[sub_test], rf_raw.predict(X_raw_s[sub_test]))

print(f"{'Model':<30} {'R²':>8}")
print("-" * 40)
for name, r2 in results.items():
    print(f"{name:<30} {r2:>8.4f}")

Model                                R²
----------------------------------------
LR raw (16 feat)                 0.3143
LR unweighted (6D)               0.2203
RF unweighted (6D)               0.4672
LR pm25-weighted (6D)            0.2392
RF pm25-weighted (6D)            0.4748
RF raw (16 feat)                 0.5400


## Latent space comparison

Visualize both latent spaces side by side, colored by PM2.5.

In [9]:
np.random.seed(42)
n_plot = 50_000
pidx = np.random.choice(len(y), n_plot, replace=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

seasons = pd.to_datetime(df['date'].values).month.map(
    lambda m: {12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',
               6:'Summer',7:'Summer',8:'Summer',9:'Fall',10:'Fall',11:'Fall'}[m]).values
season_colors = {'Winter':'#1f77b4','Spring':'#2ca02c','Summer':'#ff7f0e','Fall':'#d62728'}

for col, (name, label) in enumerate([('latent6_nodoy','Unweighted'), ('latent6_pm25wt','PM2.5-weighted')]):
    Z = runs[name]['z_mu'][pidx]
    
    # Color by PM2.5
    ax = axes[0, col]
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=y[pidx], s=0.5, alpha=0.3, cmap='YlOrRd', rasterized=True)
    plt.colorbar(sc, ax=ax, label='log(PM2.5+1)')
    ax.set_title(f'{label} — log(PM2.5)')
    ax.set_xlabel('z1'); ax.set_ylabel('z2')
    
    # Color by season
    ax = axes[1, col]
    for s in ['Winter','Spring','Summer','Fall']:
        m = seasons[pidx] == s
        ax.scatter(Z[m, 0], Z[m, 1], c=season_colors[s], s=0.5, alpha=0.3, label=s, rasterized=True)
    ax.legend(markerscale=10, fontsize=8)
    ax.set_title(f'{label} — Season')
    ax.set_xlabel('z1'); ax.set_ylabel('z2')

plt.tight_layout()
plt.savefig('plots/latent_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Latent dim correlations with PM2.5

In [10]:
# Correlation of each latent dim with log(PM2.5) for both runs
cidx = np.random.choice(len(y), 200_000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, label) in zip(axes, [('latent6_nodoy','Unweighted'), ('latent6_pm25wt','PM2.5-weighted')]):
    Z = runs[name]['z_mu'][cidx]
    corrs = [np.corrcoef(Z[:, d], y[cidx])[0, 1] for d in range(6)]
    
    bars = ax.bar(range(6), corrs)
    for i, (bar, r) in enumerate(zip(bars, corrs)):
        bar.set_color('#d62728' if r > 0 else '#1f77b4')
        ax.text(i, r + 0.01 * np.sign(r), f'{r:.3f}', ha='center', fontsize=9)
    ax.set_xticks(range(6))
    ax.set_xticklabels([f'z{i}' for i in range(6)])
    ax.set_ylabel('Correlation with log(PM2.5)')
    ax.set_title(f'{label}')
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylim(-0.6, 0.6)

plt.suptitle('Latent Dim Correlation with log(PM2.5)', fontsize=13)
plt.tight_layout()
plt.savefig('plots/latent_pm25_corr.png', dpi=150, bbox_inches='tight')
plt.show()